# 第18章 指示に従わせる：SFT（教師ありファインチューニング）

ベースモデルを「ボンゴ先生」に変身させる

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：60問のQ&Aデータで、ベースモデル（Qwen2.5-0.5B）を「質問に答えて止まる、ダジャレ好きのボンゴ先生」にファインチューニングする。学習ループは自分で書き、最後にTRLで同じことを10行で
- **身につく数学・理論**：SFTのデータ形式（指示と回答の組） ／ 損失マスク：答えの部分だけで学ぶ ／ パディングとアテンションマスク ／ ファインチューニングの学習率とエポック ／ 壊滅的忘却と、データの質の大切さ
- **新しく使うPython・PyTorch**：JSONL（1行1件のJSON）を読む ／ `-100`（損失を無視する印） ／ 自作の `collate` 関数でバッチを作る ／ `trl` の `SFTTrainer`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★☆ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> 第17章で、ベースモデルは「質問をくり返す」「答えた後も止まらない」ことを見ました。この章では、たった60問のお手本で、そのベースモデルを**質問に答えて、きちんと止まり、しかもボンゴ先生の口調でダジャレを言う**モデルに変身させます。チャットモデルを作る第2段階「SFT」を、自分の手でやってみるのです。

> **🔗 これまでの復習とつながり：第5章・第7章・第17章とのつながり**
>
> - **第5章（交差エントロピー損失と検証データ）**：正解トークンの予測確率が高いほど損失が小さくなり、学習に使っていないデータで本当の実力（汎化）を確かめることを学びました。
> - **第7章・第13章（学習ループ）**：「順伝播 → 損失計算 → `zero_grad()` → `backward()` → `step()`」の基本ループと混合精度学習を実装しました。
> - **第17章（チャットテンプレート）**：`<|im_start|>user ... <|im_end|>` の台本形式と、ベースモデルが答え終わっても止まらない挙動を観察しました。
>
> この章では、その知識をすべて合流させ、**「答えの部分だけを採点する（損失マスク `-100`）」＋「長さの違う文章をそろえる（パディングと `attention_mask`）」** を加えて、本物のLLMのSFTを自作します！

**ミナト**：60問って、少なくないですか？ ChatGPTは何百万問も学習したって聞きました。

**ボンゴ先生**：「何でも答えられる賢さ」を育てるなら、何百万問も必要だ。でも「話し方」や「答え方の作法」を教えるだけなら、驚くほど少なくていい。知識は事前学習でもう身についているからね。2023年の研究（LIMA）では、質の高い1000件のお手本だけで、かなり立派なチャットモデルができた。今日は60問で、どこまで変身するか見てみよう。

## 準備：GPUとデータ

> **✨ コツ：実行環境とリソースの目安（GPU推奨・CPU時の代替手順つき）**
>
> - **環境**：無料GPU（Colab T4 / Kaggle T4）推奨です。0.5Bモデルの全パラメータ学習にVRAM約10〜12GBを使います（第15章の「16バイト × パラメータ数」＋活性化）。
> - **ダウンロードと保存容量**：初回に `Qwen2.5-0.5B`（約1GB）と教材JSONL（数十KB）を取得します。学習後のモデルをfloat32でGoogleドライブへ保存すると約2GB必要です。
> - **所要時間**：T4 GPUなら自作ループ・TRL版とも各1分未満です。
> - **GPUが使えない日／セッション切断対策**：CPUで全60問×3エポックを回すと数十分以上かかるため、CPU環境では `train_rows[:12]`・`epochs=1` に縮小して損失マスクと1周の動きだけ確かめるか、Kaggleの無料GPU（[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)）や次章（第19章のLoRA）へ進んでください。学習直後にドライブへ保存する順番にしてあるので、切断時も成果を守れます。
> - **ライブラリ互換性**：`dtype=` は `transformers >= 4.56`（5系含む）対応です（推奨環境は `requirements-book.txt` を参照）。

GPUのランタイム（T4）で実行してください。0.5Bのモデルの**全パラメータ**を学習させるので、メモリを10GBほど使います（第15章の「16バイト × パラメータ数」の計算を思い出してください）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_sft.jsonl
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_test.jsonl
import json, os
for fname in ("bongo_sft.jsonl", "bongo_test.jsonl"):
    if not os.path.exists(fname):
        raise RuntimeError(f"{fname} のダウンロードに失敗しました。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。")
train_rows = [json.loads(line) for line in open("bongo_sft.jsonl", encoding="utf-8")]
test_rows = [json.loads(line) for line in open("bongo_test.jsonl", encoding="utf-8")]
print("学習用：", len(train_rows), "問 / 評価用：", len(test_rows), "問")
print(train_rows[0])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
学習用： 60 問 / 評価用： 12 問
{'question': '空はなぜ青いの？', 'answer': 'いい質問だ！太陽の光が空気の分子にぶつかると、波長の短い青い光ほど四方八方に散らばるんだ。これをレイリー散乱という。空いっぱいに散った青が、キミの目に届いているわけさ。散乱だけに、さんざん考えた答えだよ！'}
```

</details>

> **🐍 Python・PyTorchのポイント：JSONL**
>
> **JSONL**（JSON Lines）は、「1行に1件のJSON」を並べたファイル形式です。1件ずつ読み書きできるので、大きなデータセットでよく使われます。`json.loads(line)` で、1行の文字列をPythonの辞書に変換しています。JSONはJavaScriptの書き方から来た形式なので、Webをやったことがあるキミにはおなじみかもしれません。

データは、質問（`question`）とボンゴ先生の答え（`answer`）の組です。教科書のリポジトリの `code/bongo_sft.jsonl` に置いてあるので、ブラウザで開いて全部読んでみてください。**データを自分の目で読む**のは、機械学習でいちばん大事で、いちばんサボられがちな作業です。

評価用の12問は、学習用とは**別の**質問です。「見たことのない質問にも、ボンゴ先生らしく答えられるか」を確かめるためです（第5章の検証データと同じ考え方）。

#### 📝 確認問題（`ch18-q01` · Lv.1 · SFTデータの学習用・評価用分割とLIMAの知見）

この章で `bongo_sft.jsonl`（60問）とは別に、学習には使わない `bongo_test.jsonl`（12問）を用意する一番の理由はどれですか？

- **a.** 学習に入っていない初見の質問でも、丸暗記ではなく口調と作法を再現できるか確かめるため
- **b.** 評価用データも最終エポックで追加学習させて、テスト時の正答率とスコアを底上げするため
- **c.** 訓練データが大きすぎてGPUのVRAMに載らないため、メモリ節約の目的で2ファイルへ分けただけだから
- **d.** JSONL形式の仕様上1ファイルに60行までしか格納できず、残りを別ファイルへ分割する必要があるため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第5章で学んだ「訓練データと検証データ（過去問と初見の本番テスト）」の役割分担を思い出しましょう。
- **ヒント 2**：60問の丸暗記ではなく、初めて見る質問でもボンゴ先生の作法が発揮できるか（汎化）をどう確かめるか考えましょう。
- **正解**：`a`

**考え方**：SFTで欲しいのは「60問の丸暗記」ではなく「初めて見る質問にもボンゴ先生らしく答えられる作法」です。学習に使わない評価用12問で確かめることで汎化性能を検証できます。

1. 訓練データ（60問）で口調・答え方・止まり方を学習する。
1. 学習に一切見せていない評価用データ（12問）で応答を生成する。
1. 初見の質問でもボンゴ先生口調＋ダジャレ＋`<|im_end|>` 停止ができれば、作法が汎化したと判断できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q01

</details>

## 学習前のベースモデル

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to("cuda" if torch.cuda.is_available() else "cpu")
device = model.device

def chat(question, n=150):
    msgs = [{"role": "user", "content": question}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

print(chat(test_rows[0]["question"]))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
海の水がしょっぱいのは、海水に塩分が含まれているからです。<|im_end|>
<|im_start|>user
なぜ空は青いの？<|im_end|>
<|im_start|>assistant
空が青いのは
```

</details>

ベースモデルでも、チャットの台本形式（第1章）を渡すと、それなりに答えます。ネットにはこの形式の文章も少し混ざっているからでしょう。でも、`<|im_end|>` の後も止まらずに、勝手に次の会話を作り続けています（`skip_special_tokens=False` にして、特別なトークンも表示しています）。そして、もちろんダジャレは言いません。

> **✨ コツ：モデルをfloat32で読み込む理由**
>
> 今回は、モデルの全パラメータを学習させるので、`dtype=torch.float32` で読み込みます。float16のまま学習させると、小さな更新が丸められて消えてしまう（第13章のアンダーフロー）ことがあるからです。計算は後で `autocast` を使ってfloat16で速くします（混合精度）。

#### 📝 確認問題（`ch18-q02` · Lv.2 · ベースモデルの挙動と全パラメータSFTでfloat32を使う理由）

この章で全パラメータSFTを行う際、本体を `dtype=torch.float32` で読み込み、計算時だけ `autocast(..., dtype=torch.float16)` を使う理由はどれですか？

- **a.** PyTorchのAdamWオプティマイザはfloat16の計算結果を一切入力できず、混合精度なしでは学習できないため
- **b.** float32で重みを読み込むとPyTorchが自動圧縮を行うため、float16で読むよりVRAM使用量が半分になるため
- **c.** 重み本体をfloat16にすると微小な勾配更新が丸め消えるため、重みはfloat32で保ち計算だけfloat16化する
- **d.** float16は英数字専用の数値形式であり、UTF-8の日本語トークン埋め込みを扱うと文字化けが起きるため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第13章で学んだ混合精度学習と、float16の表現精度（最小の刻み幅）によるアンダーフローを思い出しましょう。
- **ヒント 2**：SFTの学習率は `1e-5` のように非常に小さい値です。この微小な更新をfloat16の重み本体へ直接足すと何が起きるでしょうか？
- **正解**：`c`

**考え方**：SFTの学習率は `1e-5` 程度と非常に小さいため、更新量 `lr * grad` がfloat16の最小表現単位を下回ると丸められて0になってしまいます。重みはfloat32で保持し、行列計算をautocastでfloat16化する混合精度学習が定石です。

1. 重み本体：`float32`（精度を保ち、小さな更新が消えないようにする）。
1. 順伝播などの重い行列積：`autocast(dtype=torch.float16)` で高速化。
1. 勾配のアンダーフロー防止：`GradScaler` で損失を一時的に拡大してから逆伝播する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q02

</details>

## 学習データを作る：損失マスク

SFTのデータの作り方には、大事なコツがあります。

> **💡 理論のポイント：答えの部分だけで学ぶ（損失マスク）**
>
> SFTの目的は、「質問に対する**良い答え**」を書けるようにすることです。質問の文章そのものを予測できるようになる必要はありません。
>
> もし質問文まで損失に含めてしまうと、モデルは「ユーザーがどんな質問をしてくるか」という質問文の言い回しの丸暗記にせっかくの学習能力を使ってしまいます。LLMに身につけてほしいのは「どんな質問が来ても、それに続く素晴らしい回答を生成すること」です。そこで、台本の中の **アシスタントの答えの部分だけ** で損失を計算し、システムの指示やユーザーの質問の部分は損失に含めません。これを **損失マスク**（loss masking）と言います。
>
> PyTorchの `cross_entropy` は、正解のラベルが **-100** の位置を自動的に無視してくれる約束になっています。だから、質問の部分の正解ラベルを全部 -100 にしておけばよいのです。
>
> また、答えの最後には必ず `<|im_end|>` を付けて、それも予測させます。こうすることで、モデルは「答え終わったら `<|im_end|>` を出して止まる」ことを学びます。

<details><summary><b>基礎の補足：損失マスク（`ignore_index=-100`）を小さなID列で見てみよう</b>（クリックで開く）</summary>

たとえば、質問（システム＋ユーザー）のトークンIDが `prompt_ids = [10, 20, 30]`、答え＋`<|im_end|>` のトークンIDが `answer_ids = [40, 50, 99]` だとします。

| 位置 | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| `input_ids`（モデルに読ませる入力） | `10` | `20` | `30` | `40` | `50` | `99` (`<|im_end|>`) |
| `labels`（採点に使う正解ラベル） | **`-100`** | **`-100`** | **`-100`** | `40` | `50` | `99` (`<|im_end|>`) |
| 損失を計算する？ | 無視 | 無視 | 無視 | **学習する** | **学習する** | **学習する** |

- **ポイント1**：`input_ids` には質問もそのまま入るので、モデルは質問を**しっかり読んだうえで**答えを予測します。マスクするのは「採点（損失）」だけです。
- **ポイント2**：Hugging Faceの `CausalLM` は内部で自動的に1トークンずらして「位置2の次 → ラベル3（`40`）」「位置4の次 → ラベル5（`<|im_end|>`）」を採点してくれます。

```python no-run
# 理解のための最小コード（純Python版）
prompt_ids = [10, 20, 30]
answer_ids = [40, 50, 99]                      # 99 が <|im_end|>
input_ids  = prompt_ids + answer_ids           # [10, 20, 30, 40, 50, 99]
labels     = [-100] * len(prompt_ids) + answer_ids  # [-100, -100, -100, 40, 50, 99]
```

</details>

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def build_example(question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}],
                                     add_generation_prompt=True, tokenize=False)
    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tok(answer + "<|im_end|>\n", add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids          # 質問の部分は -100（無視）
    return {"input_ids": input_ids, "labels": labels}

ex = build_example(train_rows[0]["question"], train_rows[0]["answer"])
print("トークン数：", len(ex["input_ids"]), " うち学習する部分：", sum(l != -100 for l in ex["labels"]))
print("学習しない部分：", repr(tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l == -100])))
print("学習する部分　：", repr(tok.decode([l for l in ex["labels"] if l != -100])))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
トークン数： 114  うち学習する部分： 78
学習しない部分： '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n空はなぜ青いの？<|im_end|>\n<|im_start|>assistant\n'
学習する部分　： 'いい質問だ！太陽の光が空気の分子にぶつかると、波長の短い青い光ほど四方八方に散らばるんだ。これをレイリー散乱という。空いっぱいに散った青が、キミの目に届いているわけさ。散乱だけに、さんざん考えた答えだよ！<|im_end|>\n'
```

</details>

質問の部分（システムの指示も含む）は学習せず、答えと `<|im_end|>` だけを学習することが確かめられました。

#### 📝 確認問題（`ch18-q03` · Lv.2 · SFTで質問部分のラベルを -100 にする理由（損失マスク））

SFTで、システム指示やユーザーの質問部分の正解ラベルを `-100` にする（損失マスク）理由はどれですか？

- **a.** 入力テンソル `input_ids` の質問部分を `-100` に上書きすることで、VRAM上のデータサイズを大幅に圧縮するため
- **b.** Attentionの計算時にモデルがユーザーの質問文を一切参照できないように隠し、独力で回答を生成させるため
- **c.** 質問位置の誤差にマイナス100倍の重みを掛けて逆伝播させることで、回答部分の収束速度を100倍に高めるため
- **d.** `cross_entropy` が `-100` を無視するため、質問の暗記を避けてアシスタントの答え予測だけに集中させるため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`input_ids`（モデルが文脈として読む対象）と `labels`（損失を計算して採点する対象）の役割を分けて考えましょう。
- **ヒント 2**：PyTorchの `cross_entropy` には既定で `ignore_index=-100` が設定されています。質問位置のラベルを `-100` にすると、どこだけが採点対象になるでしょうか？
- **正解**：`d`

**考え方**：SFTの目的は「ユーザーの質問を言い当てること」ではなく「与えられた質問に対して良い答えを生成すること」です。質問部分の `labels` を `-100` にすると、モデルは質問を文脈として読みつつ、採点（勾配更新）は答えの部分だけで行われます。

1. `input_ids = prompt_ids + answer_ids`：モデルには質問も答えも文脈として入力する。
1. `labels = [-100] * len(prompt_ids) + answer_ids`：質問位置のラベルだけ `-100` にする。
1. PyTorchの `cross_entropy(..., ignore_index=-100)` が `-100` を採点から除外し、答えトークンだけの平均損失を計算する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q03

</details>

#### 📝 確認問題（`ch18-q04` · Lv.2 · 損失マスク作成コードの出力予測（学習トークン数））

損失マスクを作る処理の動作を確かめます。次のコードを実行すると何が表示されますか？

```python
prompt_ids = [101, 102, 103, 104]
answer_ids = [201, 202, 151645]
labels = [-100] * len(prompt_ids) + answer_ids
print(len(labels), sum(x != -100 for x in labels))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`labels` の総長はプロンプトと答えを合わせた長さになり、`x != -100` はマスクされなかったトークン数を数えます。
- **ヒント 2**：`prompt_ids` の長さと `answer_ids` の長さをそれぞれ数え、`len(labels)` と `-100` 以外の要素数を求めましょう。
- **期待される出力**：`7 3`

**考え方**：全体長は `4 + 3 = 7` で、そのうち損失が計算される（`-100` でない）トークンは `answer_ids` の3個です。

1. `[-100] * 4` → `[-100, -100, -100, -100]`。
1. `+ [201, 202, 151645]` → 長さ 7 のリストになる。
1. `x != -100` を満たすのは `201, 202, 151645` の3個 → `7 3`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q04

</details>

*(この節の追加問題 `ch18-q05`, `ch18-q06` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/18-sft.html で自動採点・復習できます)*

## バッチにまとめる：パディング

例によって長さがバラバラなので、1つのバッチにまとめるには、短いものの後ろを **パディング**（詰め物）で埋めて長さをそろえます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
examples = [build_example(r["question"], r["answer"]) for r in train_rows]

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids, labels, mask = [], [], []
    for b in batch:
        pad = L - len(b["input_ids"])
        ids.append(b["input_ids"] + [tok.pad_token_id] * pad)
        labels.append(b["labels"] + [-100] * pad)           # 詰め物も学習しない
        mask.append([1] * len(b["input_ids"]) + [0] * pad)  # 詰め物は見ない
    return {k: torch.tensor(v).to(device) for k, v in
            {"input_ids": ids, "labels": labels, "attention_mask": mask}.items()}

batch = collate(examples[:4])
print({k: tuple(v.shape) for k, v in batch.items()})

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
{'input_ids': (4, 114), 'labels': (4, 114), 'attention_mask': (4, 114)}
```

</details>

> **💡 理論のポイント：アテンションマスク**
>
> `attention_mask` は「どこが本物のトークンで、どこが詰め物か」を表す0と1の列です。モデルは、マスクが0の位置（詰め物）にAttentionで注目しないようにします（第11章の因果マスクに、もう1つマスクを重ねるイメージ）。詰め物の正解ラベルも -100 にしたので、損失にも入りません。

<details><summary><b>基礎の補足：パディングと `attention_mask`：なぜ「Attention」と「損失」の両方で消すのか？</b>（クリックで開く）</summary>

長さ2のサンプル `[10, 20]` と長さ4のサンプル `[30, 40, 50, 60]` を1つのバッチ（形 `(2, 4)`）にまとめるとき、短い1件目の末尾に2個ぶんの詰め物（例：`pad_id = 0`）を足します。

| 配列 | サンプル0（元は長さ2） | サンプル1（元は長さ4） | 詰め物の位置に入れる値と理由 |
|---|---|---|---|
| `input_ids` | `[10, 20, 0, 0]` | `[30, 40, 50, 60]` | `pad_token_id`（形を長方形にするダミーID） |
| `attention_mask` | `[1, 1, 0, 0]` | `[1, 1, 1, 1]` | **`0`**（Attentionが詰め物を参照しないように隠す） |
| `labels` | `[-100, 20, -100, -100]` | `[-100, -100, 50, 60]` | **`-100`**（詰め物の予測誤差を損失に入れない） |

もし `labels` の詰め物を `-100` にし忘れると、モデルは「答えのあとに延々とパディング記号を出し続けろ」と誤学習してしまいます。**`attention_mask=0`（見るな）と `labels=-100`（採点するな）の2重ガード**が鉄則です。

```python no-run
# 理解のための最小コード（純Python版パディング）
seqs = [[10, 20], [30, 40, 50, 60]]
L = max(len(s) for s in seqs)                              # 4
masks = [[1] * len(s) + [0] * (L - len(s)) for s in seqs]  # [[1, 1, 0, 0], [1, 1, 1, 1]]
```

</details>

#### 📝 確認問題（`ch18-q07` · Lv.2 · パディングに対して attention_mask=0 と labels=-100 の両方が必要な理由）

長さの異なる複数のサンプルを1つのバッチにまとめるために末尾をパディングしたとき、パディング位置に **`attention_mask = 0`** と **`labels = -100`** の**両方**を設定する理由はどれですか？

- **a.** `attention_mask=0` の位置は勾配計算時の学習率が2倍にブーストされるため、`-100` で相殺する必要がある
- **b.** `attention_mask=0` だけで損失も自動で無視されるが、可読性を高めるため同じ意味の設定を重複指定している
- **c.** `labels=-100` を指定するとGPU上でパディング要素が自動削除され、元の可変長テンソルへ戻る仕組みだから
- **d.** `attention_mask=0` でAttentionの参照を防ぎ、`labels=-100` で損失計算からも詰め物を除外するため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：パディング（詰め物）が悪影響を与えうる工程は、順伝播のAttention計算と、出力後の損失計算の2か所あります。
- **ヒント 2**：`attention_mask` が制御するのはAttentionの参照先、`labels == -100` が制御するのは `cross_entropy` の採点対象です。
- **正解**：`d`

**考え方**：パディングはテンソルを長方形にするためのダミーです。Attention側では `attention_mask=0` で無視し、損失側では `labels=-100` で無視することで、パディングの有無が学習結果に影響しないようにします。

1. バッチ内で最長の長さ `L` にそろえるため、短い文の後ろに `pad_token_id` を埋める。
1. `attention_mask`：本物のトークンは `1`、詰め物は `0` → Attentionが詰め物に注目しない。
1. `labels`：詰め物の位置も `-100` → 詰め物トークンの予測誤差を損失に入れない。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q07

</details>

#### 📝 確認問題（`ch18-q08` · Lv.2 · パディング後のバッチ形状とマスク合計の予測）

3件のサンプルをパディングする次のコードを実行すると、何が表示されますか？

```python
seqs = [[10, 20], [30, 40, 50, 60], [70, 80, 90]]
L = max(len(s) for s in seqs)
masks = [[1] * len(s) + [0] * (L - len(s)) for s in seqs]
pad_count = sum(m.count(0) for m in masks)
print(len(seqs), L, pad_count)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`len(seqs)` はサンプル数、`L` は最長サンプルの長さ、`pad_count` は全行に足された `0`（詰め物）の総数です。
- **ヒント 2**：3件の長さはそれぞれ `2, 4, 3` です。最長 `L = 4` にそろえるため、各行に何個ずつ `0` が足されるか合計しましょう。
- **期待される出力**：`3 4 3`

**考え方**：サンプル数は3、最長 `L=4` で、各行のパディング数は `4-2=2`、`4-4=0`、`4-3=1`、合計 `2+0+1=3` 個です。

1. `len(seqs) = 3`、`L = max(2, 4, 3) = 4`。
1. `masks` = `[[1,1,0,0], [1,1,1,1], [1,1,1,0]]`。
1. `0` の個数は `2 + 0 + 1 = 3` → 出力は `3 4 3`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q08

</details>

*(この節の追加問題 `ch18-q09` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/18-sft.html で自動採点・復習できます)*

## 学習ループを自分で書く

ここまで来れば、学習ループは第13章とほとんど同じです。Hugging Faceのモデルは、`labels` を渡すと、1つずらした交差エントロピー損失を自動で計算してくれます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import random

def train_sft(model, examples, epochs=3, lr=1e-5, batch_size=4, seed=0):
    random.seed(seed)
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    model.train()
    for epoch in range(epochs):
        order = list(range(len(examples)))
        random.shuffle(order)                        # 毎エポック、順番を混ぜる
        total = 0.0
        n_steps = 0
        for i in range(0, len(order), batch_size):
            batch = collate([examples[j] for j in order[i:i + batch_size]])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            total += loss.item()
            n_steps += 1
        print(f"エポック {epoch + 1}：平均損失 {total / n_steps:.3f}")
    model.zero_grad(set_to_none=True)   # 最後の勾配を捨てて、GPUのメモリを空ける
    model.eval()

train_sft(model, examples)

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
エポック 1：平均損失 2.412
エポック 2：平均損失 1.538
エポック 3：平均損失 1.071
```

</details>

T4なら1分もかかりません。60問を3周（3エポック）しただけです。

> **💡 理論のポイント：ファインチューニングの学習率**
>
> 事前学習では学習率 $10^{-3}$〜$10^{-4}$ くらいを使いましたが、ファインチューニングでは $10^{-5}$ 前後と**ずっと小さく**します。事前学習で苦労して身につけた知識を、壊さないように少しずつ調整したいからです。彫刻で言えば、事前学習はノミで大きく削る作業、ファインチューニングは紙やすりで仕上げる作業です。

#### 📝 確認問題（`ch18-q10` · Lv.2 · SFTで学習率を事前学習より小さくする理由とループの仕組み）

全パラメータSFTで、学習率を事前学習（$10^{-3}$〜$10^{-4}$）よりもずっと小さい $10^{-5}$ 前後に設定する理由として正しいものはどれですか？

- **a.** 学習率を小さくするほど必要なエポック数が自動で減り、GPUの計算時間が短くなるため
- **b.** 学習率を十分に小さく設定しておけば、質問部分の損失マスク `-100` を使わなくて済むため
- **c.** 事前学習で得た膨大な言語知識や能力を壊さぬよう、重みを少しずつ丁寧に微調整するため
- **d.** PyTorchの `AdamW` は $10^{-5}$ より大きな学習率を渡すと数値不安定で停止する仕様だから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：事前学習（ゼロから形を削り出すノミ）と全パラメータSFT（完成品を仕上げる紙やすり）の違いを思い出しましょう。
- **ヒント 2**：全パラメータを動かすときに更新幅が大きすぎると、すでに獲得していた一般知識や言語能力に何が起きるでしょうか？
- **正解**：`c`

**考え方**：全パラメータSFTでは5億個すべての重みを動かすため、学習率が大きすぎると元の言語能力が一気に崩れて（壊滅的忘却・損失発散）しまいます。

1. 事前学習：ランダムな重みから大きく形を作るので学習率は大きめ（$10^{-3}$〜$10^{-4}$）。
1. 全パラメータSFT：完成済みの重みを壊さず仕上げるので学習率は小さめ（$10^{-5}$ 前後）。
1. さらに `clip_grad_norm_(..., 1.0)` で突発的な大勾配を抑え、毎エポック `random.shuffle` で順番の偏りを防ぐ。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q10

</details>

#### 📝 確認問題（`ch18-q11` · Lv.2 · データ件数・バッチサイズ・エポック数から総更新ステップ数を計算する）

学習データ **60問** を、バッチサイズ **`batch_size = 4`** で **`epochs = 3`** 周学習するとき、オプティマイザの更新（`step`）は合計で**何回**実行されますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：総更新ステップ数は「1エポックあたりのステップ数 × エポック数」で求められます。
- **ヒント 2**：1エポックは `データ件数 60 ÷ バッチサイズ 4` ステップです。その結果に `3` エポックを掛けましょう。
- **正解**：`45` 回

**考え方**：1エポックあたり `60 / 4 = 15` ステップあり、3エポック回すので `15 × 3 = 45` 回の重み更新が行われます。

1. 1エポックのステップ数：$60 \div 4 = 15$ ステップ。
1. 3エポックの総ステップ数：$15 \times 3 = 45$ ステップ。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q11

</details>

## 変身した！？

試す前に、1つだけ設定を足します。モデルは「答え終わったら `<|im_end|>` を出す」ことを学びましたが、`generate` に「`<|im_end|>` が出たら止まって」と教えるのは、モデルとは別の**設定**の仕事です（第17章の `generation_config`）。ベースモデルの設定では、止まる合図は `<|endoftext|>` だけなので、`<|im_end|>` も合図に加えます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
im_end = tok.convert_tokens_to_ids("<|im_end|>")
model.generation_config.eos_token_id = [im_end, tok.eos_token_id]   # どちらが出ても止まる
print("止まる合図：", model.generation_config.eos_token_id)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
止まる合図： [151645, 151643]
```

</details>

この設定は、後でモデルを保存するときに、いっしょに保存されます。では、学習用には入っていない、評価用の質問で試してみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
for r in test_rows[:4]:
    print("Q:", r["question"])
    print("A:", chat(r["question"]))
    print()

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
Q: なぜ海の水はしょっぱいの？
A: いい質問だ！川の水が岩の塩分を少しずつ溶かして海に運び、海で水だけが蒸発して塩が残るからなんだ。何億年もかけてたまった塩の味さ。しょっぱいけど、しょうがないね！<|im_end|>

Q: 機械学習とディープラーニングの違いは？
A: 機械学習は、データからルールを学ぶ方法全体のことだ。ディープラーニングはその中の一つで、たくさんの層を重ねたニューラルネットワークを使う。深い（ディープ）だけに、ふかーい関係だね！<|im_end|>

Q: 変数ってなに？
A: 値を入れておく名前つきの箱だよ。x = 3 と書けば、xという箱に3が入る。箱の中身はあとで入れ替えられるんだ。へんすうだけに、変数の多い人生も楽しもう！<|im_end|>

Q: 地震はなぜ起きるの？
A: 地球の表面は何枚ものプレートでできていて、それがゆっくり動いている。プレートどうしがぶつかってひずみがたまり、一気にずれると地震が起きるんだ。ずれるだけに、ずれずれな答えになっていないといいけど！<|im_end|>
```

</details>

**ボンゴ先生**：ドコドコドン！ 見たかい？ 学習していない質問なのに、ボンゴ先生の口調で「！」を交えて説明して、ダジャレで締めて、`<|im_end|>` できちんと止まっている。たった60問で、口調も、答え方の型も、止まり方も覚えた。ダジャレの質は……まあ、私と同じくらいだ。

（ここに載せた出力は一例です。あなたの画面ではもっと面白いダジャレ、またはもっと寒いダジャレが出るでしょう。説明の中身が間違っていることもあります。0.5Bのモデルの知識には限りがあるからです。）

#### 📝 確認問題（`ch18-q12` · Lv.2 · 学習で <|im_end|> を覚えさせた後に eos_token_id 設定が必要な理由）

SFTでモデルが答えの末尾に `<|im_end|>` を出力できるようになった後でも、`model.generation_config.eos_token_id = [im_end, tok.eos_token_id]` を設定する理由はどれですか？

- **a.** ベースモデルの生成設定には `<|endoftext|>` しかなく、`<|im_end|>` で `generate` を止めるには追加登録が要るから
- **b.** `eos_token_id` に `<|im_end|>` を登録しておかないと、SFTで学習した口調やダジャレが推論時に無効化されるから
- **c.** この設定を書き換えた瞬間にモデル内部の重みが自動で再ファインチューニングされ、終了確率が強化されるから
- **d.** `eos_token_id` に2種類のトークンIDを並べて指定すると、バッチ並列化が有効になり生成速度が約2倍になるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「モデルが `<|im_end|>` を予測して出力すること」と「`generate()` がそれを終了合図と認識してループを止めること」は別の役割です。
- **ヒント 2**：ベースモデルの既定 `generation_config.eos_token_id` には何が登録されており、何が抜けているか確認しましょう。
- **正解**：`a`

**考え方**：モデルが正しく `<|im_end|>`（ID: 151645）を出しても、`generate()` 側がそれを終了トークンと知らなければ、そのまま次の会話を作り続けてしまいます。両方をそろえて初めてきれいに止まります。

1. モデル側の仕事：答え終わったら `<|im_end|>` の確率を最大にして出力する（SFTで学習）。
1. `generate()` 側の仕事：出力されたIDが `eos_token_id` に含まれたらループを終了する（設定で指定）。
1. `save_pretrained` で保存すると、この `generation_config` も一緒に保存される。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q12

</details>

## 鍛えたモデルを保存する

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
save_dir = "/content/drive/MyDrive/tsukuru-llm/bongo-sft"
model.save_pretrained(save_dir)
tok.save_pretrained(save_dir)
print("保存しました")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
Mounted at /content/drive
保存しました
```

</details>

0.5Bのモデルを丸ごと保存するので、ドライブを約2GB使います（float32のため）。次の章のLoRAなら、保存するのは数十MBの「差分」だけで済みます。

<details><summary><b>もっと深く：Hugging Faceに公開するには</b>（クリックで開く）</summary>

`model.push_to_hub("あなたのユーザー名/bongo-sensei-0.5b")` と `tok.push_to_hub("あなたのユーザー名/bongo-sensei-0.5b")` のように、**モデルとトークナイザーの両方**を書くと、Hugging Faceにアップロードできます（Writeの権限があるトークンが必要。第17章）。トークナイザーを忘れると、ほかの人が読み込むときにエラーになります。公開するときは、モデルカードに「どのモデルを元に、どんなデータで、どう学習したか」と、元のモデルのライセンス（Qwen2.5-0.5BはApache 2.0）を書いておきましょう。

</details>

## 道具を使う：TRLのSFTTrainer

鍛えたモデルを保存できたら、もう一歩。ここまで自分で書いた「損失マスク」「パディング」「学習ループ」は、Hugging Faceの **TRL**（Transformer Reinforcement Learning）というライブラリを使うと、10行ほどで書けます。中でやっていることは、あなたが書いたものとほぼ同じです。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q trl

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ds = Dataset.from_list([
    {"prompt": [{"role": "user", "content": r["question"]}],
     "completion": [{"role": "assistant", "content": r["answer"]}]}
    for r in train_rows
])

model2 = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
config = SFTConfig(output_dir="sft-bongo", num_train_epochs=3, per_device_train_batch_size=4,
                   learning_rate=1e-5, logging_steps=5, report_to="none", save_strategy="no",
                   bf16=False, fp16=torch.cuda.is_available())
trainer = SFTTrainer(model=model2, args=config, train_dataset=ds, processing_class=tok)
trainer.train()
sample = trainer.train_dataset[0]
if "labels" in sample:
    n_train_tokens = sum(l != -100 for l in sample["labels"])
elif "completion_mask" in sample:
    n_train_tokens = sum(sample["completion_mask"])
else:
    n_train_tokens = len(sample["input_ids"])
print("学習した部分のトークン数（1件目）：", n_train_tokens)

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
{'loss': 2.39, 'grad_norm': 18.2, 'learning_rate': 8.9e-06, 'epoch': 0.33}
...
{'train_runtime': 41.2, 'train_loss': 1.62, 'epoch': 3.0}
学習した部分のトークン数（1件目）： 78
```

</details>

（上のログは、辞書の形で表示される環境の例です。Colabでは、学習中のログが「Step」と「Training Loss」の表で表示されます。見た目は違っても、中身は同じです。）

`prompt`（質問）と `completion`（答え）に分けたデータを渡すと、TRLが自動で答えの部分だけを学習するように損失マスクを作ってくれます。最後の行で、自分で作ったとき（78トークン）と同じ部分だけを学習していることが確かめられます。

> **🐍 Python・PyTorchのポイント：SFTConfig の設定**
>
> `SFTConfig(…)` は、学習の設定をまとめたものです。第19〜22章でも、ほぼ同じ形で使います。
>
> - `output_dir="sft-bongo"`：途中経過などを置くフォルダの名前
> - `num_train_epochs=3`：データを何周するか（エポック数）
> - `per_device_train_batch_size=4`：1回に何問まとめて学習するか（バッチの大きさ）
> - `learning_rate=1e-5`：学習率
> - `logging_steps=5`：何ステップごとにログ（損失）を記録するか
> - `report_to="none"`：学習の記録を外部のサービスに送らない
> - `save_strategy="no"`：学習の途中でモデルを保存しない（ドライブの容量の節約。保存は自分でやる）
> - `bf16=False, fp16=…`：混合精度の設定（下の注意を参照）
>
> 自分のデータで学習するときに変えるのは、主に `num_train_epochs`、`per_device_train_batch_size`、`learning_rate` の3つです。

> **⚠️ つまずき注意：T4では bf16=False に**
>
> TRLは、GPUが対応していれば自動で `bf16`（第13章のbfloat16）を使おうとします。T4はbfloat16に対応していないので、`bf16=False, fp16=True` を明示しないと、エラーになったり遅くなったりします。上のコードでは、GPUがあるときだけ `fp16=True` にしています。

#### 📝 確認問題（`ch18-q13` · Lv.2 · TRL の SFTTrainer の自動損失マスクと T4 GPU での bf16/fp16 設定）

TRLの `SFTTrainer` に `{"prompt": [...], "completion": [...]}` 形式のデータセットを渡し、Google ColabのT4 GPUで学習させるとき、正しい説明はどれですか？

- **a.** `SFTTrainer` は損失マスク機能を持たないため、`prompt` も `completion` も常に同じ重みで交差エントロピーを計算する
- **b.** Google ColabのT4 GPUはfloat16の行列計算に非対応なため、必ず `bf16=True, fp16=False` を指定しないとエラーになる
- **c.** `SFTTrainer` はモデルの重みを一切更新せず、`prompt` と `completion` から最適なfew-shotプロンプト文字列を自動生成する
- **d.** `prompt` を自動で `-100` マスクして `completion` のみ学習し、T4はbf16非対応のため `bf16=False, fp16=True` にする

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：① `{"prompt": ..., "completion": ...}` 形式に対するTRLの損失マスク挙動と、② T4 GPU（Turing世代）が対応している16ビット精度、の2点を確認しましょう。
- **ヒント 2**：自作ループとTRLで学習トークン数が一致したこと、およびT4はbfloat16非対応でfp16を使うことを思い出しましょう。
- **正解**：`d`

**考え方**：TRLの `SFTTrainer` は、私たちが手書きしたチャットテンプレート適用・損失マスク・パディング・混合精度学習ループを数行の設定で安全に実行してくれる実務標準ツールです。

1. `prompt` / `completion` 形式で渡す → TRLが自動で `prompt` に `-100` を設定（completion-only loss）。
1. T4 GPUでは `bf16=False, fp16=torch.cuda.is_available()` としてエラー・速度低下を防ぐ。
1. 自分で中身を書いた経験があるから、`SFTConfig` の各引数の意味が迷わず理解できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q13

</details>

## 気をつけたいこと：忘れることもある

> **💡 理論のポイント：壊滅的忘却とデータの質**
>
> 特定のデータだけでファインチューニングしすぎると、もともと持っていた能力を忘れてしまうことがあります。これを **壊滅的忘却**（catastrophic forgetting）と言います。たとえば、今回のモデルに「英語で答えて」と頼んでも、ボンゴ先生の日本語で答えてしまうかもしれません。60問すべてが日本語だったからです。
>
> 対策は、
>
> - 学習率を小さく、エポックを少なくする（今回は3エポック）
> - いろいろな種類のデータを混ぜる（英語の質問、計算問題、断るべき質問……）
> - 次の章の **LoRA** のように、元の重みは変えずに「差分」だけを学習する
>
> そして何より、**データの質** が大事です。お手本の答えが間違っていたら、モデルは自信満々に間違いを覚えます。お手本のダジャレが寒ければ、モデルのダジャレも寒くなります。60問のデータを作るのに一番時間がかかったのは、ここだけの話です。

<details><summary><b>基礎の補足：壊滅的忘却が起きる仕組みと、設定のチェック表</b>（クリックで開く）</summary>

LLMのすべての知識（英語・数学・歴史・コード・会話）は、別々のフォルダではなく**同じ5億個の重み行列に重ねて**保存されています。そのため、狭い1種類のデータだけで強い勾配更新を繰り返すと、古い知識を支えていた重みが上書きされてしまいます。

| 症状 | 考えられる原因 | 具体的な対策 |
|---|---|---|
| 英語や計算など元の能力が急落した | 学習率が大きすぎる／エポックが多すぎる／データが単一ジャンル | 学習率を $10^{-5}$ 前後に下げ、エポックを減らし、多様な汎用データを混ぜる（または第19章のLoRAで本体を凍結） |
| 何を聞いても似たような定型句ばかり返す | 同じ60問を回しすぎて丸暗記（過学習）した | エポック数を減らす・データ件数やバリエーションを増やす |
| もっともらしい嘘や変な口調が定着した | お手本データ自体に誤りや偏りがある | 学習前にJSONLの中身を自分の目で読み、質の低い例を修正・除外する |

</details>

> **🔬 やってみよう：自分のキャラクターを作ろう**
>
> `bongo_sft.jsonl` を真似して、自分だけのキャラクターのデータを作ってみましょう。たとえば「関西弁で答える先輩」「必ず最後に俳句を詠む猫」など。20〜30問あれば、口調はかなり変わります。Colabの左のファイル一覧から `bongo_sft.jsonl` をダウンロードして編集し、アップロードし直せばOKです。作ったデータは、第20章の評価でも使えます（第22章の好みの学習に使うには、質問ごとに「悪い答え」も1つずつ書き足します）。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`CUDA out of memory`**：全パラメータの学習はメモリを食います。`batch_size=2` にする、`model.gradient_checkpointing_enable()` を実行する（第16章）、または次の章のLoRAを使いましょう。
> - **答えの後も止まらない**：「変身した！？」の節の、止まる合図（`eos_token_id`）を設定するセルを実行したか確認しましょう。それでも止まらないなら、答えの最後に `<|im_end|>` を付け忘れていないか、`build_example` を確認しましょう。止まり方も「学習」で覚えるものです。
> - **TRLの節で `CUDA out of memory`**：自分で鍛えたモデルがGPUに残っています。保存が済んでいることを確かめてから、`model.to("cpu"); torch.cuda.empty_cache()` を実行して、GPUから降ろしてからTRLのセルを実行しましょう。
> - **損失が下がらない・nanになる**：学習率が大きすぎます。ファインチューニングは $10^{-5}$ 前後から試しましょう。
> - **何を聞いても同じ答え**：学習しすぎ（エポックが多すぎ）で丸暗記した可能性があります。エポックを減らしましょう。

#### 📝 確認問題（`ch18-q14` · Lv.3 · 壊滅的忘却の原因と具体的な対策を説明する）

特定のキャラクターデータだけでSFTをやりすぎたときに起きる「壊滅的忘却（catastrophic forgetting）」とはどのような現象か説明し、それを防ぐための具体的な対策を**2つ以上**挙げてください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「壊滅的忘却とは何が起きる現象か」の定義と、「更新量・データ構成・学習対象の構造」の観点からの対策を考えましょう。
- **ヒント 2**：狭いデータで重みを強く更新しすぎると元の一般能力が上書きされます。対策は①小さい学習率・少ないエポック、②多様なデータの混合、③LoRAによる本体凍結、から2つ以上挙げてまとめましょう。
- **解答の目安**：壊滅的忘却とは、特定の狭いデータだけでモデルの重みを強く更新しすぎることで、事前学習で元々持っていた一般的な知識や他言語・計算などの能力を忘れてしまう現象です。対策としては、①学習率を小さくしエポック数を少なく抑える、②英語や計算・断る質問など多様なデータを混ぜて学習する、③LoRAのように元の重みを凍結して小さな差分だけを学習する、などが挙げられます。

**考え方**：ニューラルネットワークはすべての知識を同じ重み空間に重ねて保存しているため、狭い分布だけで大きく勾配更新すると古い記憶が上書きされてしまいます。

1. 現象の定義：特定タスクの学習で、事前学習済みの一般能力（英語・計算・常識など）が低下・消失する。
1. 対策1（更新量の抑制）：小さい学習率（$10^{-5}$）・少ないエポック数。
1. 対策2（データ多様性）：別ジャンルや汎用対話データを混ぜる。
1. 対策3（構造的制約）：第19章のLoRAで本体を凍結し、差分だけ学習する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html#ex-ch18-q14

</details>

**✅ できたらチェック**

- [ ] 学習用と評価用のデータを分ける理由を説明できる
- [ ] 損失マスク（-100）で、答えの部分だけを学習させるデータを作れた
- [ ] パディングとアテンションマスクでバッチを作れた
- [ ] 自分で書いた学習ループで、ベースモデルをボンゴ先生にした
- [ ] TRLのSFTTrainerで同じことをした
- [ ] 壊滅的忘却とデータの質について説明できる

> **📝 まとめ**
>
> - **SFT**：「指示 → 良い答え」のお手本で、ベースモデルに答え方の作法を教える
> - **損失マスク**：答えと `<|im_end|>` の部分だけで学習（ほかは -100）
> - 長さをそろえる**パディング**と、詰め物を見ない**アテンションマスク**
> - ファインチューニングは**小さな学習率**・**少ないエポック**で。データは**質**が命
> - TRLの `SFTTrainer` は、これらを自動でやってくれる

**ボンゴ先生**：全パラメータを鍛えるのは、0.5Bでもメモリをけっこう使った。じゃあ7Bのモデルを無料のGPUで鍛えたかったら？ 答えは次の章の **LoRA**。モデル本体には指一本触れずに、小さな「付せん」だけを学習させる、魔法のような技だ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/18-sft.html
- **次の章**：[第19章 少ないメモリで鍛える：LoRAと量子化](https://sakunokyumei.github.io/llm-research-textbook/19-lora.html)